# Appliances Energy Prediction: Posterior Overlap, Observed U-Statistic, and Model-Based Thresholds

This notebook reproduces the Section 4.4 (Appliances Energy Prediction) analysis of the
thesis, mirroring the California Housing workflow: well-specified threshold, observed
discrepancy under standard Bayes, beta* search, and downstream performance (Tables 7-10).


## 1. Data Loading

Appliances Energy Prediction (UCI id=374): 19735 observations at 10-minute intervals. Drop `date`, `rv1`, `rv2` (random noise variables). Remove collinear features (|r| > 0.95) from the temperature/humidity sensor pairs. Target is appliance energy consumption (Wh) — heteroscedastic (residual variance varies across time blocks) -> genuine misspecification for a Gaussian linear model.


In [1]:
import sys, importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from sklearn.preprocessing import StandardScaler

THESIS_PATH = Path('/Users/alya57/Desktop/thesis')
if str(THESIS_PATH) not in sys.path:
    sys.path.insert(0, str(THESIS_PATH))

import nig_beta_overlap_utils as _nbo
importlib.reload(_nbo)
from nig_beta_overlap_utils import (
    fit_standard_nig_posterior,
    sample_from_nig_params,
    gvi_beta_posterior_continuation,
    u_statistic_hellinger_within,
    hellinger_nig_closed_form,
    nig_predictive_params,
    predictive_coverage,
    predictive_mlpd,
    predictive_crps_student,
    nig_joint_predictive,
    minus2logBC_mvn,
)

# ── Global configuration ─────────────────────────────────────────────────────
SEED           = 888877432
RNG            = np.random.default_rng(SEED)
K_BLOCKS       = 8
N_TEST_PRED    = 200     # small fixed test set for predictive discrepancy

# NIG prior
A0, B0, SIGMA0 = 0.5, 0.5, 2.0

# GVI
BETA_EPOCHS    = 120
BETA_LR        = 0.005

# NOTE: coarser than the 5e-4 used in the synthetic-data notebooks -- validated
# to give <2% relative change in the resulting Hellinger/BC discrepancies while
# running an order of magnitude faster at this dataset scale, which is what
# makes the beta* search and downstream sweeps (Tables 8-10) tractable.
BETA_MAX_STEP  = 0.005

# U-statistic IS budget
N_IS_U         = 2500

# Well-specified threshold
N_MC_THRESHOLD = 1000

print('Configuration loaded.')
print(f'  K_BLOCKS={K_BLOCKS}, N_TEST_PRED={N_TEST_PRED}')
print(f'  Prior: a0={A0}, b0={B0}, sigma0={SIGMA0}')


Configuration loaded.
  K_BLOCKS=8, N_TEST_PRED=200
  Prior: a0=0.5, b0=0.5, sigma0=2.0


In [2]:
from ucimlrepo import fetch_ucirepo

# Load Appliances Energy Prediction (19735 x 26 before cleaning)
energy   = fetch_ucirepo(id=374)
df_energy = energy.data.features.copy()
y_raw    = energy.data.targets.values.astype(float).ravel()

# Drop date string and random noise variables
df_energy = df_energy.drop(columns=['date', 'rv1', 'rv2'])

# Remove collinear features (|r| > 0.95) -- many temp/humidity sensors are highly correlated
CORR_THRESHOLD = 0.95
X_tmp    = df_energy.values.astype(float)
corr_mat = np.abs(np.corrcoef(X_tmp.T))
upper    = np.triu(corr_mat, k=1)
drop_idx = set()
for i in range(upper.shape[0]):
    for j in range(i + 1, upper.shape[1]):
        if j not in drop_idx and upper[i, j] > CORR_THRESHOLD:
            drop_idx.add(j)
keep_idx  = [i for i in range(len(df_energy.columns)) if i not in drop_idx]
kept_cols = [df_energy.columns[i] for i in keep_idx]
X_raw     = X_tmp[:, keep_idx]
feature_names = kept_cols

print(f'Features after collinearity removal ({len(feature_names)}): {feature_names}')
print(f'X={X_raw.shape},  y={y_raw.shape}')
print(f'Target range: [{y_raw.min():.1f}, {y_raw.max():.1f}],  mean={y_raw.mean():.1f}')

# Standardise features and target
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_all = x_scaler.fit_transform(X_raw)
y_all = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

print(f'\nAfter standardisation: X mean~{X_all.mean():.3f}, std~{X_all.std():.3f}')
print(f'                       y mean~{y_all.mean():.3f}, std~{y_all.std():.3f}')
n_block = (len(y_all) - N_TEST_PRED) // K_BLOCKS
print(f'\nExpected: n/block~{n_block}, D={len(feature_names)}, n/D~{n_block/len(feature_names):.0f}x')


Features after collinearity removal (24): ['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint']
X=(19735, 24),  y=(19735,)
Target range: [10.0, 1080.0],  mean=97.7

After standardisation: X mean~0.000, std~1.000
                       y mean~0.000, std~1.000

Expected: n/block~2441, D=24, n/D~102x


## 2. Hold-out Test Set and Block Construction


In [3]:
# ── Test set + random K-block split ─────────────────────────────────────────
TEST_SEED = 19   # controls the test/train split independently of the block shuffle

rng_test      = np.random.default_rng(TEST_SEED)
test_pred_idx = rng_test.choice(len(y_all), size=N_TEST_PRED, replace=False)
X_test_pred   = X_all[test_pred_idx]
y_test_pred   = y_all[test_pred_idx]

train_idx = np.setdiff1d(np.arange(len(y_all)), test_pred_idx)
rng_block = np.random.default_rng(SEED + 100)
train_idx = rng_block.permutation(train_idx)

block_sizes = np.full(K_BLOCKS, len(train_idx) // K_BLOCKS)
block_sizes[: len(train_idx) % K_BLOCKS] += 1

blocks, ptr = [], 0
for sz in block_sizes:
    idx = train_idx[ptr: ptr + sz]
    blocks.append((X_all[idx], y_all[idx]))
    ptr += sz

D = blocks[0][0].shape[1]

print(f'Total N: {len(y_all)}')
print(f'Test set : {X_test_pred.shape}  (TEST_SEED={TEST_SEED})')
print(f'Training pool: {len(train_idx)} obs split into {K_BLOCKS} random blocks')
for k, (Xb, yb) in enumerate(blocks):
    print(f'  Block {k}: n={len(yb)}, D={D}')


Total N: 19735
Test set : (200, 24)  (TEST_SEED=19)
Training pool: 19535 obs split into 8 random blocks
  Block 0: n=2442, D=24
  Block 1: n=2442, D=24
  Block 2: n=2442, D=24
  Block 3: n=2442, D=24
  Block 4: n=2442, D=24
  Block 5: n=2442, D=24
  Block 6: n=2442, D=24
  Block 7: n=2441, D=24


## 3. Well-Specified Threshold

Fit a reference NIG to the full training pool. Simulate pairs of datasets under that model and compute the expected Hellinger discrepancy — this is the target that β* must match.


In [4]:
print('Fitting reference NIG to full training pool ...')
p_ref = fit_standard_nig_posterior(X_all, y_all, a0=A0, b0=B0, sigma0=SIGMA0)

# Use one block as the fixed design matrix for replications
n_rep = len(blocks[0][1])
X_rep = blocks[0][0]

print(f'Simulating {N_MC_THRESHOLD} pairs (n_rep={n_rep}) ...')
theta_ref = sample_from_nig_params(p_ref, n_samples=N_MC_THRESHOLD, seed=SEED + 200)
rng_thr   = np.random.default_rng(SEED + 300)
h_thr_vals = []

for m in range(N_MC_THRESHOLD):
    sigma2_m = float(theta_ref[m, 0])
    beta_m   = theta_ref[m, 1:]
    y1_rep = X_rep @ beta_m + np.sqrt(sigma2_m) * rng_thr.standard_normal(n_rep)
    y2_rep = X_rep @ beta_m + np.sqrt(sigma2_m) * rng_thr.standard_normal(n_rep)
    p1 = fit_standard_nig_posterior(X_rep, y1_rep, a0=A0, b0=B0, sigma0=SIGMA0)
    p2 = fit_standard_nig_posterior(X_rep, y2_rep, a0=A0, b0=B0, sigma0=SIGMA0)
    h_m, _ = hellinger_nig_closed_form(p1, p2)
    h_thr_vals.append(h_m)
    if (m + 1) % 200 == 0:
        print(f'  {m+1}/{N_MC_THRESHOLD} ...')

h_thr_vals = np.array(h_thr_vals)
thr_mean   = float(np.mean(h_thr_vals))
thr_se     = float(np.std(h_thr_vals, ddof=1) / np.sqrt(N_MC_THRESHOLD))
thr_q95    = float(np.quantile(h_thr_vals, 0.95))

print(f'\nWell-specified threshold: {thr_mean:.6f} +/- {thr_se:.6f}  (q95={thr_q95:.6f})')


Fitting reference NIG to full training pool ...
Simulating 1000 pairs (n_rep=2442) ...
  200/1000 ...


  400/1000 ...
  600/1000 ...


  800/1000 ...
  1000/1000 ...

Well-specified threshold: 12.418096 +/- 0.107508  (q95=18.428040)


## 4. Observed Discrepancy Under Standard Bayes

Fit the Standard NIG on each block and compute the observed U-statistic. This should sit *above* the well-specified threshold, confirming misspecification and motivating the β* search.


In [5]:
# Observed discrepancy under Standard Bayes  (Table 7)
print(f'Fitting Standard NIG on {K_BLOCKS} blocks ...')
std_params = []
for k, (Xb, yb) in enumerate(blocks):
    p = fit_standard_nig_posterior(Xb, yb, a0=A0, b0=B0, sigma0=SIGMA0)
    std_params.append(p)

u_std, _ = u_statistic_hellinger_within(std_params, n_is=N_IS_U)

print(f'Well-specified threshold : {thr_mean:.6f} +/- {thr_se:.6f}')
print(f'Observed U-stat (Std)    : {u_std:.6f}')
print(f'Gap (Std - threshold)    : {u_std - thr_mean:+.6f}')
print()
print('Standard Bayes is above the threshold -> model is misspecified.')
print('beta* search will find beta such that U(beta*) matches the threshold.')


Fitting Standard NIG on 8 blocks ...
Well-specified threshold : 12.418096 +/- 0.107508
Observed U-stat (Std)    : 21.358041
Gap (Std - threshold)    : +8.939945

Standard Bayes is above the threshold -> model is misspecified.
beta* search will find beta such that U(beta*) matches the threshold.


## 5. β* Search

Find β* such that U(β*) matches the well-specified threshold. Coarse grid scan → brentq.


In [6]:
def fit_beta_blocks(beta_val):
    out = []
    for Xb, yb in blocks:
        p_raw = gvi_beta_posterior_continuation(
            Xb, yb, beta_div=float(beta_val), a0=A0, b0=B0, sigma0=SIGMA0,
            n_epochs=BETA_EPOCHS, lr=BETA_LR, verbose=False, max_step=BETA_MAX_STEP,
        )
        out.append({
            'mu':   np.asarray(p_raw['mu']),
            'V':    np.asarray(p_raw['V']),
            'd':    float(p_raw['d']),
            'beta': float(p_raw['beta']),
        })
    return out

def observed_u(beta_val):
    params_b = fit_beta_blocks(beta_val)
    u, _     = u_statistic_hellinger_within(params_b, n_is=N_IS_U)
    return float(u), params_b

def gap(beta_val):
    u, _ = observed_u(beta_val)
    return u - thr_mean

# Coarse scan
beta_grid_coarse = np.linspace(1.02, 1.16, 8)
gaps_coarse      = []

print(f'Coarse scan  (threshold = {thr_mean:.6f})')
print(f'{"beta":>8}  {"U_beta":>10}  {"gap":>10}')
print('-' * 34)

for bv in beta_grid_coarse:
    u_b, _ = observed_u(bv)
    g      = u_b - thr_mean
    gaps_coarse.append(g)
    print(f'{bv:>8.4f}  {u_b:>10.6f}  {g:>+10.6f}')

gaps_coarse  = np.array(gaps_coarse)
sign_changes = np.where(np.diff(np.sign(gaps_coarse)))[0]

if len(sign_changes) > 0:
    idx = int(sign_changes[0])
    lo  = float(beta_grid_coarse[idx])
    hi  = float(beta_grid_coarse[idx + 1])
    print(f'Sign change in [{lo:.4f}, {hi:.4f}] — running brentq ...')
    beta_star = float(brentq(gap, lo, hi, xtol=1e-3, rtol=1e-3, maxiter=15))
else:
    best_idx  = int(np.argmin(np.abs(gaps_coarse)))
    beta_star = float(beta_grid_coarse[best_idx])
    print('No sign change — using closest grid point.')

u_star, beta_star_params = observed_u(beta_star)
print(f'beta*     = {beta_star:.4f}')
print(f'U(beta*)  = {u_star:.6f}')
print(f'threshold = {thr_mean:.6f}')
print(f'gap       = {u_star - thr_mean:+.6f}')


Coarse scan  (threshold = 12.418096)
    beta      U_beta         gap
----------------------------------


  1.0200   18.066547   +5.648451


  1.0400   15.561905   +3.143809


  1.0600   13.662601   +1.244505


  1.0800   12.404995   -0.013101


  1.1000   11.449686   -0.968411


  1.1200   10.678975   -1.739121


  1.1400   10.131345   -2.286751


  1.1600    9.761118   -2.656978
Sign change in [1.0600, 1.0800] — running brentq ...


beta*     = 1.0800
U(beta*)  = 12.404995
threshold = 12.418096
gap       = -0.013101


## 6. Downstream Performance

Stability of parameter inference (Table 8), stability of predictive inference (Table 9), and
predictive accuracy (Table 10), for Standard Bayes and beta=1.1/beta*/1.3/1.5, across the
K_BLOCKS training blocks.


In [7]:
# Parameter-inference stability across blocks (Table 8), for Standard Bayes and
# beta=1.1/beta*/1.3/1.5.
from itertools import combinations

BETA_MODELS = [
    ('Standard Bayes', None),
    ('beta=1.1', 1.1),
    (f'beta*={beta_star:.2f}', beta_star),
    ('beta=1.3', 1.3),
    ('beta=1.5', 1.5),
]

pairs = list(combinations(range(K_BLOCKS), 2))
all_block_params = {}

print(f"{'Model':<16}  {'Mean L2':>9}  {'Max L2':>9}  {'Mean Fro':>10}  {'Max Fro':>10}  {'Mean Hell':>11}  {'Max Hell':>11}")
print('-' * 82)

for label, bval in BETA_MODELS:
    if bval is None:
        params = std_params
    else:
        params = fit_beta_blocks(bval)
    all_block_params[label] = params

    l2s   = [np.linalg.norm(params[i]['mu'] - params[j]['mu']) for i, j in pairs]
    fros  = [np.linalg.norm(params[i]['V'] - params[j]['V'], ord='fro') for i, j in pairs]
    hells = [hellinger_nig_closed_form(params[i], params[j])[0] for i, j in pairs]

    print(f"{label:<16}  {np.mean(l2s):9.4f}  {np.max(l2s):9.4f}  "
          f"{np.mean(fros):10.4f}  {np.max(fros):10.4f}  "
          f"{np.mean(hells):11.4f}  {np.max(hells):11.4f}")


Model               Mean L2     Max L2    Mean Fro     Max Fro    Mean Hell     Max Hell


----------------------------------------------------------------------------------
Standard Bayes       0.5789     0.7920      0.0045      0.0060      21.3580      30.9137


beta=1.1             0.3117     0.4519      0.0061      0.0083      11.4498      21.0594


beta*=1.08           0.3553     0.5083      0.0056      0.0075      12.4050      22.2889


beta=1.3             0.2502     0.3547      0.0136      0.0274       8.5844      12.1284


beta=1.5             0.2709     0.3999      0.0235      0.0457       7.1402      10.0892


In [8]:
# Predictive-inference stability across blocks (Table 9): joint -2 log BC on
# the fixed held-out test set, for the same 5 models.

print(f"Joint predictive -2 log BC over {len(y_test_pred)} test points")
print()
print(f"{'Model':<16}  {'Mean -2logBC':>13}  {'Max -2logBC':>12}  {'Min -2logBC':>12}")
print('-' * 60)

for label, _ in BETA_MODELS:
    params = all_block_params[label]
    joints = [nig_joint_predictive(params[k], X_test_pred) for k in range(K_BLOCKS)]
    discs  = [minus2logBC_mvn(*joints[i], *joints[j]) for i, j in pairs]
    discs  = np.array(discs)
    print(f"{label:<16}  {np.mean(discs):13.4f}  {np.max(discs):12.4f}  {np.min(discs):12.4f}")


Joint predictive -2 log BC over 200 test points

Model              Mean -2logBC   Max -2logBC   Min -2logBC
------------------------------------------------------------


Standard Bayes           1.6165        2.4904        0.7897


beta=1.1                 1.0384        1.7724        0.6610


beta*=1.08               1.0954        1.8411        0.6709


beta=1.3                 1.1612        1.7641        0.7170


beta=1.5                 1.2144        1.8869        0.7581


In [9]:
# Leave-one-block-out CV predictive metrics (MLPD, RMSE, Cov90/95/99, CRPS) for
# the 5 models reported in Table 10.

K = len(blocks)
table10_rows = []

for label, bval in BETA_MODELS:
    fold_metrics = []
    for k in range(K):
        X_tr = np.vstack([blocks[j][0] for j in range(K) if j != k])
        y_tr = np.concatenate([blocks[j][1] for j in range(K) if j != k])
        X_te, y_te = blocks[k]

        if bval is None:
            p = fit_standard_nig_posterior(X_tr, y_tr, a0=A0, b0=B0, sigma0=SIGMA0)
        else:
            p_raw = gvi_beta_posterior_continuation(
                X_tr, y_tr, beta_div=float(bval), a0=A0, b0=B0, sigma0=SIGMA0,
                n_epochs=BETA_EPOCHS, lr=BETA_LR, verbose=False, max_step=BETA_MAX_STEP,
            )
            p = {'mu': np.asarray(p_raw['mu']), 'V': np.asarray(p_raw['V']),
                 'd': float(p_raw['d']), 'beta': float(p_raw['beta'])}

        df_pred, loc, scale = nig_predictive_params(p, X_te)
        mlpd = predictive_mlpd(df_pred, loc, scale, y_te)
        rmse = float(np.sqrt(np.mean((y_te - loc) ** 2)))
        cov90 = predictive_coverage(df_pred, loc, scale, y_te, level=0.90)
        cov95 = predictive_coverage(df_pred, loc, scale, y_te, level=0.95)
        cov99 = predictive_coverage(df_pred, loc, scale, y_te, level=0.99)
        crps = predictive_crps_student(df_pred, loc, scale, y_te)
        fold_metrics.append((mlpd, rmse, cov90, cov95, cov99, crps))

    arr = np.array(fold_metrics)
    mean = arr.mean(axis=0)
    se = arr.std(axis=0, ddof=1) / np.sqrt(K)
    table10_rows.append((label, mean, se))
    print(f'  done: {label}')

header = f"{'Model':<22s}  {'MLPD':>8s}  {'RMSE':>7s}  {'Cov90':>6s}  {'Cov95':>6s}  {'Cov99':>6s}  {'CRPS':>7s}"
print()
print('=' * len(header))
print(header)
print('-' * len(header))
for label, mean, se in table10_rows:
    print(f"{label:<22s}  {mean[0]:8.4f}  {mean[1]:7.4f}  {mean[2]:6.3f}  {mean[3]:6.3f}  {mean[4]:6.3f}  {mean[5]:7.4f}")
    print(f"{'(SE)':<22s}  {se[0]:8.4f}  {se[1]:7.4f}  {se[2]:6.3f}  {se[3]:6.3f}  {se[4]:6.3f}  {se[5]:7.4f}")
    print('-' * len(header))


  done: Standard Bayes


  done: beta=1.1


  done: beta*=1.08


  done: beta=1.3


  done: beta=1.5

Model                       MLPD     RMSE   Cov90   Cov95   Cov99     CRPS
--------------------------------------------------------------------------
Standard Bayes           -1.3336   0.9177   0.937   0.947   0.961   0.4342
(SE)                      0.0129   0.0117   0.001   0.001   0.001   0.0033
--------------------------------------------------------------------------
beta=1.1                 -1.6637   0.9236   0.902   0.917   0.934   0.4033
(SE)                      0.0332   0.0114   0.002   0.002   0.001   0.0036
--------------------------------------------------------------------------
beta*=1.08               -1.5315   0.9218   0.910   0.923   0.940   0.4059
(SE)                      0.0276   0.0114   0.002   0.001   0.001   0.0036
--------------------------------------------------------------------------
beta=1.3                 -2.2856   0.9304   0.875   0.897   0.917   0.4031
(SE)                      0.0537   0.0111   0.002   0.002   0.001   0.0037
-------